In [ ]:
%pip install statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, chi2_contingency, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
df = pd.read_csv("Titanic-Dataset.csv")
print(df.shape)

In [ ]:
alpha = 0.05

def decide(p):
    return "Reject H0" if p < alpha else "Fail to reject H0"

print(df["Sex"].value_counts())
print(df["Pclass"].value_counts())

In [ ]:
# Welch's t-test: Age of male vs female
g1 = df.loc[df["Sex"] == "male", "Age"].dropna()
g2 = df.loc[df["Sex"] == "female", "Age"].dropna()
print(len(g1), g1.mean(), g1.std())
print(len(g2), g2.mean(), g2.std())

plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="Sex", y="Age")
plt.title("Comparison of Age by Sex")
plt.tight_layout()
plt.show()

t_stat, t_p = ttest_ind(g1, g2, equal_var=False)
print(t_stat, t_p, decide(t_p))

In [ ]:
# Chi-square test: Sex vs Survived
observed = pd.crosstab(df["Sex"], df["Survived"])

plt.figure(figsize=(6, 4))
sns.heatmap(observed, annot=True, fmt="d", cmap="Blues")
plt.title("Contingency Table (Sex vs Survived)")
plt.tight_layout()
plt.show()

chi2, chi_p, dof, expected = chi2_contingency(observed)
print(chi2, dof, chi_p, decide(chi_p))

expected_df = pd.DataFrame(expected, index=observed.index, columns=observed.columns)
print(expected_df.round(2))

In [ ]:
# One-way ANOVA: Fare across Pclass
print(df.groupby("Pclass")["Fare"].agg(["count", "mean"]))

plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="Pclass", y="Fare")
plt.title("Fare across Pclass")
plt.tight_layout()
plt.show()

samples = [df.loc[df["Pclass"] == c, "Fare"] for c in [1, 2, 3]]
f_stat, anova_p = f_oneway(*samples)
print(f_stat, anova_p, decide(anova_p))

In [ ]:
# Tukey HSD post-hoc test
tukey = pairwise_tukeyhsd(df["Fare"], df["Pclass"], alpha=alpha)
print(tukey)

In [ ]:
results = pd.DataFrame({
    "Test": ["Welch's t-test", "Chi-square test", "One-way ANOVA"],
    "Statistic": [t_stat, chi2, f_stat],
    "p-value": [t_p, chi_p, anova_p]
})
results["Decision"] = results["p-value"].apply(decide)
print(results)
results.to_csv("hypothesis_testing_results.csv", index=False)